# Operaciones de aprendizaje automático
## Maestría en Inteligencia Artificial Aplicada

### TC5061 | Proyecto MiBici: propuesta y base reproducible | Colaborativo

#### **Equipo:** 7
#### **Nombre:** Luis Arturo Sánchez Davila
#### **Matrícula:** A01840576

## Objetivo
Notebook de **análisis exploratorio (EDA)** del proyecto MiBici para la pregunta **P3 (clustering)**: cuantifica la calidad de los datos, el volumen y los patrones en el tiempo, y la demanda y el balance por estación, sobre el dataset de viajes **sin modificarlo**. Cada hallazgo queda documentado para que el equipo acuerde su tratamiento en la limpieza (`0.03-lasd-limpieza-p3`).

## Entradas y salidas
- **Entrada:** `data/interim/viajes_AAAA-MM_AAAA-MM.parquet` (viajes normalizados y sin limpiar, que deja `dataset.construir`; ver `0.01-lasd-exploracion-carga`) y la nomenclatura vigente de `data/raw/estaciones/`.
- **Salida:** ninguna; los hallazgos se documentan en las observaciones de cada sección.

## Nota de reproducibilidad
El notebook se ejecuta desde la carpeta `notebooks/`. Solo se muestran **agregados**; no se imprimen filas con `usuario_id`. Las cifras corresponden a la corrida del 15-oct-2026 con los 81 meses del rango.

## Serie de notebooks de P3
| Notebook | Contenido | Entrega |
|---|---|---|
| `0.01-lasd-exploracion-carga` | Paso 1: carga y descripción | Dataset normalizado en `data/interim/` |
| `0.02-lasd-eda-p3` | Paso 2: análisis exploratorio | Hallazgos de calidad, tiempo y estaciones |
| `0.03-lasd-limpieza-p3` | Paso 3: limpieza, partición y preprocesamiento | `data/processed/` (viajes de modelado, estaciones y demanda) |
| `0.04-lasd-baseline-p3` | Paso 4 (4.1 a 4.4): modelo base y registro en MLflow | Modelo base y métricas (0.05 lo recalcula con el mismo código y semilla) |
| `0.05-lasd-mejoramodelo-p3` | Paso 4 (4.5 a 4.9): modelo alternativo y registro en MLflow | Comparación contra el baseline |

In [ ]:
# Importación de librerías y registro de versiones (evidencia de reproducibilidad)
from pathlib import Path
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
import seaborn as sns

print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("pyarrow     :", pyarrow.__version__)
print("matplotlib  :", matplotlib.__version__)
print("seaborn     :", sns.__version__)

## Configuración y carga

Las rutas y el rango salen de `tc5061_proyecto_equipo07/config.py`, las gráficas de `plots.py` y las variables derivadas de `features.py`. Se carga el dataset normalizado que dejó `dataset.construir`; este notebook no lo modifica.

La lógica reutilizable vive en el módulo `tc5061_proyecto_equipo07` (funciones `p3_*`); este notebook la llama y documenta los hallazgos.

In [ ]:
# Configuración compartida (el notebook se ejecuta desde notebooks/)
try:
    from tc5061_proyecto_equipo07 import dataset
except ModuleNotFoundError:  # paquete sin instalar; `make requirements` lo instala con `-e .`
    sys.path.insert(0, str(Path.cwd().parent))
    from tc5061_proyecto_equipo07 import dataset
from tc5061_proyecto_equipo07 import features, plots
from tc5061_proyecto_equipo07.config import MES_FIN, MES_INICIO

# Opciones de visualización de pandas
pd.set_option("display.max_columns", None)   # mostrar todas las columnas
pd.set_option("display.max_colwidth", None)  # no recortar textos largos

# Entrada: viajes normalizados y sin limpiar; dataset.p3_cargar_viajes los valida contra el manifiesto
viajes = dataset.p3_cargar_viajes(MES_INICIO, MES_FIN)
print(f"Registros: {len(viajes):,} | columnas: {viajes.shape[1]}")

## Paso 2: Análisis exploratorio de datos (EDA)

En este paso se explora el dataset unido **sin modificarlo**: cada hallazgo se cuantifica y se documenta para que el equipo acuerde su tratamiento en la limpieza. Las preguntas que guían el EDA salen del ML Canvas (`references/ml-canvas.md`):
- **Calidad:** ¿qué registros no se pueden usar tal cual (duplicados, duraciones, estaciones sin nomenclatura)?
- **Tiempo:** ¿cómo evoluciona el volumen y qué patrones horarios y semanales hay? Es la base de P1.
- **Estaciones:** ¿qué tan concentrada está la demanda y qué estaciones acumulan o pierden bicicletas? Es la base de P2 y P3.

Solo se muestran **agregados**; no se imprimen filas con `usuario_id`.

### 2.1 Preparación

Se cargan la nomenclatura de estaciones y tres variables derivadas: duración en minutos, hora del día y día de la semana (0 = lunes).

In [ ]:
# Variables derivadas para el EDA (las columnas originales no cambian)
features.p3_variables_de_tiempo(viajes)

# Nomenclatura de estaciones (misma detección de codificación que los viajes)
nomenclatura = dataset.p3_cargar_estaciones()

print("Nomenclatura:", len(nomenclatura), "estaciones")
print(f"Memoria del DataFrame: {viajes.memory_usage(deep=True).sum() / 1024**3:.1f} GB")
nomenclatura.describe().T

### 2.2 Calidad de los datos

Antes de buscar patrones se mide qué parte del dataset no es utilizable tal cual. Se revisan cuatro cosas:
- **Duplicados:** un mismo `viaje_id` repetido, y si la fila completa también lo está (duplicado exacto) o difiere (conflicto).
- **Duración:** negativa, cero y extrema. Un viaje de duración cero suele ser un desbloqueo fallido; uno de más de 24 horas, una bicicleta no devuelta.
- **Año de nacimiento:** solo se cuentan valores imposibles; la variable no se usará en el modelado.
- **Estaciones sin nomenclatura:** sin capacidad ni ubicación no se pueden usar en P2 ni P3.

In [ ]:
# Duplicados: identificadores repetidos y filas completas repetidas
COLUMNAS_ORIGINALES = ["viaje_id", "usuario_id", "genero", "anio_de_nacimiento",
                       "inicio_del_viaje", "fin_del_viaje", "origen_id", "destino_id"]
ids_repetidos = int(viajes["viaje_id"].duplicated().sum())
filas_repetidas = int(viajes.duplicated(subset=COLUMNAS_ORIGINALES).sum())

print(f"Registros con viaje_id ya visto      : {ids_repetidos:,}")
print(f"Filas completas repetidas            : {filas_repetidas:,}")
print(f"Ids repetidos con datos distintos    : {ids_repetidos - filas_repetidas:,}")

In [ ]:
# Duración del viaje en minutos: anomalías y percentiles
duracion = viajes["duracion_min"]
negativas = int((duracion < 0).sum())
ceros = int((duracion == 0).sum())
mas_de_24h = int((duracion > 24 * 60).sum())

print(f"Duración negativa : {negativas:,}")
print(f"Duración cero     : {ceros:,} ({ceros / len(viajes):.2%})")
print(f"Más de 24 horas   : {mas_de_24h:,}")
print(f"Máxima            : {duracion.max() / 60:.1f} horas")
duracion.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99, .999]).round(2).to_frame("duracion_min")

In [ ]:
# Distribución de la duración: viajes de hasta 60 minutos y escala logarítmica de todos los positivos
fig = plots.p3_grafica_duracion(duracion)

In [ ]:
# Año de nacimiento: solo se cuentan valores imposibles (agregados, sin filas individuales)
anio = viajes["anio_de_nacimiento"]
antes_1900 = int((anio < 1900).sum())
despues_2010 = int((anio > 2010).sum())
print(f"Anterior a 1900 : {antes_1900:,}")
print(f"Posterior a 2010: {despues_2010:,}")
print(f"Plausible (1920 a 2010): {anio.between(1920, 2010).sum() / anio.notna().sum():.2%} de los registros con dato")

In [ ]:
# Estaciones de los viajes que no están en la nomenclatura vigente
ids_nomenclatura = set(nomenclatura["id"])
ids_usados = set(viajes["origen_id"].dropna().unique()) | set(viajes["destino_id"].dropna().unique())
ids_sin_nomenclatura = sorted(ids_usados - ids_nomenclatura)

viajes_afectados = int((~viajes["origen_id"].isin(ids_nomenclatura) | ~viajes["destino_id"].isin(ids_nomenclatura)).sum())
sin_capacidad = int((nomenclatura["dpcapacity"] == 0).sum())

print("Estaciones con viajes         :", len(ids_usados))
print("Estaciones en la nomenclatura :", len(ids_nomenclatura))
print("Ids sin nomenclatura          :", ids_sin_nomenclatura)
print(f"Viajes afectados              : {viajes_afectados:,} ({viajes_afectados / len(viajes):.3%})")
print("Estaciones con capacidad 0    :", sin_capacidad)

In [ ]:
# Resumen de calidad: registros afectados por hallazgo
total = len(viajes)
resumen_calidad = pd.DataFrame({
    "hallazgo": ["Filas completas repetidas", "Duración cero", "Duración mayor a 24 h",
                 "Año de nacimiento anterior a 1900", "Año de nacimiento posterior a 2010",
                 "Origen o destino sin nomenclatura", "Género nulo", "Año de nacimiento nulo"],
    "registros": [filas_repetidas, ceros, mas_de_24h, antes_1900, despues_2010, viajes_afectados,
                  int(viajes["genero"].isna().sum()), int(viajes["anio_de_nacimiento"].isna().sum())],
})
resumen_calidad["% del total"] = (resumen_calidad["registros"] / total * 100).round(3)
resumen_calidad

**Observaciones de la calidad de los datos:**
- **Duplicados:** 758 registros repiten un `viaje_id` ya visto y los 758 son filas idénticas en todas las columnas; no hay identificadores repetidos con datos distintos. Eliminarlos no implica elegir entre versiones en conflicto.
- **Duración:** ninguna es negativa. 55,035 viajes (0.20 %) duran cero minutos y 159 duran más de 24 horas, con un máximo de 469.2 horas. La mediana es 9.45 min y el 99 % dura 32.80 min o menos; el 99.9 % no pasa de 58.72 min. El volumen extremo es mínimo, pero distorsiona la media (11.00 min) y la desviación (15.48 min) por una cola muy larga.
- **Año de nacimiento:** 1,255 valores anteriores a 1900 y 2,101 posteriores a 2010 (0.012 % del total entre ambos); el 99.99 % de los registros con dato cae entre 1920 y 2010. La variable no se usará en el modelado, por lo que el hallazgo solo importa si se describe a los usuarios.
- **Estaciones:** 475 estaciones tienen viajes y la nomenclatura vigente tiene 484. Tres identificadores (124, 405 y 407) aparecen en viajes y no en la nomenclatura: 3,961 viajes (0.015 %). Cuatro estaciones de la nomenclatura tienen capacidad 0.
- **Impacto:** ningún hallazgo supera el 0.51 % del total; el más grande son los nulos de `genero` (0.51 %). Los que obligarían a excluir registros (duplicados, duración cero o extrema y estaciones sin nomenclatura) suman como máximo 59,913 registros (0.22 %), sin descontar traslapes. La limpieza no compromete el tamaño del dataset.

### 2.3 Volumen de viajes en el tiempo

La serie mensual muestra la tendencia, el efecto de la pandemia y la continuidad de los 81 meses del rango (2023-04 y 2023-05 estuvieron retirados del portal y volvieron a publicarse). Es el primer insumo para decidir si P1 necesita un indicador de periodo y cómo calcular los rezagos.

In [ ]:
# Viajes por mes sobre el calendario completo del rango (los meses sin archivo quedan como NaN)
mensual = viajes.groupby("periodo_archivo", observed=True).size()
mensual.index = pd.PeriodIndex(mensual.index.astype(str), freq="M")
mensual = mensual.reindex(pd.period_range(MES_INICIO, MES_FIN, freq="M"))

fig = plots.p3_grafica_serie_mensual(mensual)

print("Mes con menos viajes:", mensual.idxmin(), f"({mensual.min():,.0f})")
print("Mes con más viajes  :", mensual.idxmax(), f"({mensual.max():,.0f})")
print(f"Mediana mensual     : {mensual.median():,.0f}")
print("Meses sin dato      :", [str(p) for p in mensual[mensual.isna()].index])

In [ ]:
# Viajes por año: total, promedio por mes con dato y variación contra el año anterior
anual = mensual.groupby(mensual.index.year).agg(viajes="sum", meses_con_dato="count")
anual["promedio_mensual"] = (anual["viajes"] / anual["meses_con_dato"]).round(0)
anual["variacion_vs_anio_previo_%"] = (anual["promedio_mensual"].pct_change() * 100).round(1)
anual

**Observaciones del volumen en el tiempo:**
- **Tendencia:** el promedio mensual crece de 238,839 viajes en 2020 a 387,160 en 2024 y se estabiliza: 377,669 en 2025 (−2.5 %) y 373,563 en 2026 (−1.1 %, con 9 meses). Después de 2024 el sistema muestra una meseta, no un crecimiento.
- **Pandemia:** el volumen cae de 367,864 viajes en febrero de 2020 a 129,276 en abril (−64.9 %). El primer mes que supera el nivel de febrero de 2020 es marzo de 2022 (377,505), dos años después. Un modelo entrenado con 2020 aprende un régimen que no se repite; se justifica un indicador de periodo o excluir 2020 de los ajustes.
- **Continuidad:** los 81 meses del rango tienen datos. 2023-04 (333,211) y 2023-05 (364,184) estuvieron retirados del portal el 14-oct-2026 y volvieron a publicarse al día siguiente; con ellos la serie ya no tiene hueco y los rezagos y ventanas que cruzan esos meses se pueden calcular. Abril cae 17.2 % contra marzo (402,460) y mayo se recupera. Si el portal los retira de nuevo, `MESES_SIN_DATOS` los excluye de la tabla de demanda y no se imputan.
- **Estacionalidad:** hay caídas recurrentes en diciembre (por ejemplo, 299,005 en 2021-12 y 314,164 en 2023-12), consistentes con el periodo vacacional. El mes del año es una variable relevante para P1.
- **Último mes:** 2026-09 (490,056) es el máximo de toda la serie: 24.5 % más que agosto (393,524) y 30.1 % más que septiembre de 2025 (376,789). El EDA no explica la causa; hay que verificar que sea un valor real y no una carga atípica antes de que forme parte del conjunto de prueba.

### 2.4 Patrones por hora y día de la semana

Se mide qué proporción de los viajes ocurre en cada hora y día. Un perfil con dos picos (mañana y tarde) en días laborales indica uso para ir al trabajo; es la señal que justifica incluir `hora`, `dia_semana` y festivos como features de P1 y P2.

In [ ]:
# Proporción de viajes por día de la semana y hora de inicio
conteo = viajes.groupby(["dia_semana", "hora"], observed=True).size().unstack(fill_value=0)
proporcion = conteo / conteo.to_numpy().sum() * 100

fig = plots.p3_grafica_heatmap_hora_dia(proporcion)

In [ ]:
# Perfil horario: días laborales frente a fin de semana (proporción dentro de cada grupo)
laboral = conteo.loc[0:4].sum()
fin_de_semana = conteo.loc[5:6].sum()

fig = plots.p3_grafica_perfil_horario(laboral, fin_de_semana)

print("Horas pico lunes a viernes:", laboral.nlargest(3).index.tolist())
print("Horas pico fin de semana  :", fin_de_semana.nlargest(3).index.tolist())
print(f"Viajes en fin de semana   : {fin_de_semana.sum() / (laboral.sum() + fin_de_semana.sum()):.1%}")

**Observaciones de los patrones horarios:**
- **Días laborales:** dos picos claros a las 8:00 (8.3 % de los viajes) y a las 18:00 (10.5 %), con un valle de 10:00 a 11:00 (4.2 a 4.3 %). Las horas 7, 8, 17 y 18 concentran el 32.3 % de los viajes laborales. Es el patrón de traslado al trabajo.
- **Fin de semana:** un solo pico amplio entre las 11:00 y las 14:00, con máximo a las 13:00 (8.8 %). Sábado y domingo concentran el 20.4 % de los viajes, por debajo del 28.6 % que corresponde a 2 de 7 días: el uso es predominantemente laboral.
- **Horario de servicio:** entre la 1:00 y las 4:59 hay solo 194 viajes en 27.1 millones (0.0007 %); la hora 0 tiene 145,857 (0.5 %), con más peso en sábado y domingo (31,410 y 31,698). El sistema prácticamente no opera de madrugada.
- **Implicación:** `hora`, `dia_semana` y `es_fin_de_semana` son features justificadas por evidencia. Para P1, las horas de 1:00 a 4:59 no deben rellenarse con ceros como si fueran demanda cero medida: son horas de servicio cerrado y conviene excluirlas o tratarlas aparte.

### 2.5 Demanda y balance por estación

Se mide cuántos viajes inician y terminan en cada estación. Dos preguntas:
- **Concentración:** si pocas estaciones concentran la demanda, el modelo de P1 se evaluará sobre todo en ellas.
- **Balance neto:** `llegadas − salidas`. Una estación con balance muy positivo acumula bicicletas (riesgo de quedarse sin anclajes) y una con balance muy negativo las pierde (riesgo de quedarse sin bicicletas). Es la base de P2.

In [ ]:
# Salidas, llegadas y balance neto por estación, unidas a la nomenclatura
salidas = viajes["origen_id"].value_counts()
llegadas = viajes["destino_id"].value_counts()
estaciones = pd.DataFrame({"salidas": salidas, "llegadas": llegadas}).fillna(0).astype(int)
estaciones["neto"] = estaciones["llegadas"] - estaciones["salidas"]
estaciones["neto_%"] = (estaciones["neto"] / estaciones["salidas"].clip(lower=1) * 100).round(1)
estaciones = estaciones.join(nomenclatura.set_index("id")[["name", "dpcapacity"]])

orden = estaciones["salidas"].sort_values(ascending=False)
n10 = int(np.ceil(len(orden) * 0.10))
print("Estaciones                       :", len(estaciones))
print(f"Salidas por estación (mediana)   : {estaciones['salidas'].median():,.0f}")
print(f"Salidas por estación (máxima)    : {estaciones['salidas'].max():,.0f}")
print(f"El 10% de estaciones ({n10}) concentra: {orden.head(n10).sum() / orden.sum():.1%} de los viajes")
print("Estaciones con menos de 1,000 salidas:", int((estaciones["salidas"] < 1000).sum()))
estaciones.sort_values("salidas", ascending=False).head(10)[["name", "salidas", "llegadas", "neto", "dpcapacity"]]

In [ ]:
# Concentración de la demanda y distribución del balance neto
fig = plots.p3_grafica_concentracion_balance(orden, estaciones)

In [ ]:
# Estaciones con mayor desbalance neto y relación entre demanda y capacidad
con_capacidad = estaciones[estaciones["dpcapacity"] > 0]
print("Mayor acumulación de bicis (neto positivo):")
display(estaciones.sort_values("neto", ascending=False).head(5)[["name", "salidas", "llegadas", "neto", "neto_%"]])
print("Mayor pérdida de bicis (neto negativo):")
display(estaciones.sort_values("neto").head(5)[["name", "salidas", "llegadas", "neto", "neto_%"]])

correlacion = con_capacidad["salidas"].corr(con_capacidad["dpcapacity"], method="spearman")
print(f"Correlación de Spearman entre salidas y capacidad: {correlacion:.2f}")
print(f"Estaciones con |neto| mayor al 10% de sus salidas: {int((estaciones['neto_%'].abs() > 10).sum())}")

**Observaciones de la demanda y el balance por estación:**
- **Concentración:** 475 estaciones con viajes; la mediana es 31,587 salidas por estación y el máximo 567,876 (GDL-049 Lopez Cotilla / Penitenciaría, capacidad 39). El 10 % de las estaciones (48) concentra el 37.8 % de los viajes. 74 estaciones (15.6 %) tienen menos de 1,000 salidas en seis años y nueve meses; probablemente sean estaciones nuevas o retiradas, lo que se debe verificar con su primera y última fecha activa antes de decidir su tratamiento.
- **Balance neto:** 273 de las 475 estaciones (57.5 %) tienen un balance mayor al 10 % de sus salidas, positivo o negativo, acumulado en el periodo. Las que más acumulan bicicletas están en el centro (GDL-049 con +148,627, +26.2 %; GDL-032 Degollado/Hidalgo con +32.3 %). Las que más pierden están sobre Av. Vallarta (GDL-127 con −98,030, −37.4 %; GDL-129, GDL-130 y GDL-128 con −20.1 % a −30.0 %). Es un patrón geográfico consistente, y es justo el desbalance que el rebalanceo debe compensar. El EDA no observa los traslados que ya hace la operación.
- **Capacidad:** la correlación de Spearman entre salidas y capacidad es −0.02: la capacidad instalada no sigue a la demanda. Hay estaciones de capacidad 15 con 408,005 salidas (GDL-033) y otras de capacidad 39 con 567,876. Un mismo desbalance absoluto pesa muy distinto según la estación.
- **Implicación para P1, P2 y P3:** en P1 el error debe reportarse por estación y no solo en total, porque el 10 % de las estaciones domina el volumen. En P2 el balance debe normalizarse por capacidad (`neto_sobre_capacidad`), no usarse en valores absolutos. En P3, el desbalance y la ubicación son señales de agrupamiento y las estaciones con menos de 1,000 salidas deben filtrarse antes de agrupar.

## Hallazgos que pasan a la limpieza

Los hallazgos de 2.2 (duplicados, duraciones, año de nacimiento y estaciones sin nomenclatura), de 2.3 (periodo y continuidad de la serie) y de 2.5 (estaciones con poca historia y capacidad) son la entrada de `0.03-lasd-limpieza-p3`, donde se convierten en reglas explícitas.